In [1]:
# ══════════════════════════════════════════════════════════════════════
# Downstream Tool-Choice Validation: BoR vs Fixed-K via Claude API
# ──────────────────────────────────────────────────────────────────────
# BFCL data (public). 3-seed training with std.
# FK5/FK1 Claude calls cached (deterministic). BoR varies per seed.
#
# Cost: ~$3-5 | Runtime: ~15 min training + ~15 min API calls
# ══════════════════════════════════════════════════════════════════════

ANTHROPIC_API_KEY = "key"

# ── Config ──
SEEDS = [42, 123, 456]     # 3 seeds for variance
CAND_N = 370               # full BFCL registry (matches Section 4.1)
MAX_QUERIES = 0            # 0 = use all 400 queries
TRAIN_EPS = 8000           # full training
CLAUDE_MODEL = "claude-sonnet-4-6"
MAX_TOOLS_IN_PROMPT = 30

# ── Installs ──
import sys, subprocess, importlib.util
def ensure(name, pip=None):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or name])
ensure("anthropic"); ensure("numpy"); ensure("torch")
ensure("sentence_transformers", "sentence-transformers")
ensure("sklearn", "scikit-learn")
ensure("huggingface_hub")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "sympy>=1.13.1"])

import json, math, random, time
from collections import deque, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download
import anthropic

device = "cuda" if torch.cuda.is_available() else "cpu"
assert ANTHROPIC_API_KEY.strip(), "Set ANTHROPIC_API_KEY before running."
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY.strip())

from huggingface_hub import login
login(token="YOUR_HF_TOKEN", add_to_git_credential=False)

# ══════════════════════════════════════════════════════════════════════
# PART 1: Load BFCL data
# ══════════════════════════════════════════════════════════════════════
print("[1/6] Loading BFCL data...", flush=True)
fpath = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_simple.json",
    repo_type="dataset"
)
with open(fpath) as f:
    raw = [json.loads(line) for line in f]
print(f"  Loaded {len(raw)} entries")

registry = {}
entries = []
for entry in raw:
    question = entry.get('question', [])
    query_text = ''
    if isinstance(question, list) and question:
        inner = question[0]
        if isinstance(inner, list) and inner:
            for msg in inner:
                if isinstance(msg, dict) and msg.get('role') == 'user':
                    query_text = msg.get('content', '')
                    break
        elif isinstance(inner, dict):
            if inner.get('role') == 'user':
                query_text = inner.get('content', '')
    if not query_text: continue
    funcs = entry.get('function', [])
    if not funcs or not isinstance(funcs, list): continue
    func = funcs[0]
    name = func.get('name', '')
    desc = func.get('description', '')
    if not name: continue
    params = func.get('parameters', {})
    param_names = list(params.get('properties', {}).keys()) if isinstance(params, dict) else []
    text = f"{name}: {desc}. Parameters: {', '.join(param_names)}" if param_names else f"{name}: {desc}"
    registry[name] = text
    entries.append({'query': query_text[:500], 'gt_tool': name})

tool_names_list = sorted(registry.keys())
tool_descs_list = [registry[n] for n in tool_names_list]
tool_descs_map = {n: registry[n] for n in tool_names_list}
tool_to_idx = {n: i for i, n in enumerate(tool_names_list)}
N_FULL = len(tool_names_list)

query_list = [e for e in entries if e['gt_tool'] in tool_to_idx]
# Use first seed for shuffling
random.seed(SEEDS[0]); np.random.seed(SEEDS[0]); torch.manual_seed(SEEDS[0])
if MAX_QUERIES > 0 and len(query_list) > MAX_QUERIES:
    random.shuffle(query_list)
    query_list = query_list[:MAX_QUERIES]
print(f"  Registry: {N_FULL} tools, Queries: {len(query_list)}")
assert len(query_list) > 0, "No queries loaded."

# ══════════════════════════════════════════════════════════════════════
# PART 2: Embed + score + build candidate sets (once, seed-independent)
# ══════════════════════════════════════════════════════════════════════
print("[2/6] Embedding...", flush=True)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tool_embs = encoder.encode(tool_descs_list, batch_size=64,
                           show_progress_bar=False, normalize_embeddings=True)
q_embs = encoder.encode([q["query"] for q in query_list], batch_size=64,
                        show_progress_bar=False, normalize_embeddings=True)
score_matrix = q_embs @ tool_embs.T

def difficulty_bucket(rank):
    if rank == 1: return "1_easy"
    if 2 <= rank <= 5: return "2_medium"
    if 6 <= rank <= 20: return "3_hard"
    return "4_vhard"

base_instances = []
for i, q in enumerate(query_list):
    gold_idx = tool_to_idx[q["gt_tool"]]
    ranked_global = np.argsort(-score_matrix[i])
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({"query": q["query"], "gold_idx": gold_idx,
        "scores_all": score_matrix[i], "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full, "gt_tool": q["gt_tool"]})

indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEEDS[0])

CAND_N = min(CAND_N, N_FULL)

def make_cand(base, N):
    gold_idx = base["gold_idx"]
    hard = [j for j in base["ranked_global"] if j != gold_idx][:N-1]
    cand = [gold_idx] + list(hard)
    scores = np.array([base["scores_all"][j] for j in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_ids = [cand[j] for j in order]
    ranked_sc = scores[order]
    gold_rank = int(ranked_ids.index(gold_idx) + 1)
    return {"gold_rank": gold_rank, "ranked_tool_ids": ranked_ids,
            "scores": ranked_sc, "N": N, "query": base["query"],
            "bucket": difficulty_bucket(gold_rank), "gt_tool": base["gt_tool"]}

instances = [make_cand(base_instances[i], CAND_N) for i in range(len(base_instances))]
train_insts = [instances[i] for i in train_idx]
test_insts = [instances[i] for i in test_idx]
print(f"  N={CAND_N}, Train: {len(train_insts)}, Test: {len(test_insts)}")
buckets = defaultdict(int)
for inst in test_insts: buckets[inst["bucket"]] += 1
print(f"  Buckets: {dict(sorted(buckets.items()))}")

# ══════════════════════════════════════════════════════════════════════
# PART 3: DQN infrastructure
# ══════════════════════════════════════════════════════════════════════
BATCH = 128; REPLAY_SZ = 50000; LR = 1e-3; GAMMA = 0.95; STEP_COST = 0.01
TARGET_EVERY = 500

def bor_reward(success, k, N):
    return -math.log2(max(k / N, 1e-12)) if success else 0.0

def f1_reward(success, k):
    return (2.0 / (k + 1.0)) if success else 0.0

def state_vec(inst, k):
    s = inst["scores"]; N = inst["N"]; idx = min(k-1, N-1)
    cur = float(s[idx])
    nxt = float(s[idx+1]) if idx+1 < N else float(s[idx])
    first = float(s[0]); mean = float(s.mean()); std = float(s.std() + 1e-6)
    gap = cur - nxt if idx+1 < N else 0.0
    return np.array([k/N, math.log2(k+1)/math.log2(N+1), cur, nxt, gap,
                     (cur-mean)/std, cur/(abs(first)+1e-6)], dtype=np.float32)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7,64), nn.ReLU(),
            nn.Linear(64,64), nn.ReLU(),
            nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def train_dqn(train_insts, reward_name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rfn = bor_reward if reward_name == "bor" else f1_reward
    net = QNet().to(device); tgt = QNet().to(device)
    tgt.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SZ)
    eps_s, eps_e = 1.0, 0.05; step = 0
    for ep in range(1, TRAIN_EPS+1):
        inst = random.choice(train_insts); k = 1; done = False
        eps = eps_e + (eps_s - eps_e) * max(0, 1 - ep/TRAIN_EPS)
        while not done:
            sv = state_vec(inst, k)
            if random.random() < eps: a = random.randint(0,1)
            else:
                with torch.no_grad():
                    a = int(net(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
            if k >= inst["N"]: a = 0
            if a == 0:
                success = inst["gold_rank"] <= k
                r = rfn(success, k, inst["N"]) if reward_name == "bor" else rfn(success, k)
                replay.append((sv, a, float(r), None, 1.0)); done = True
            else:
                k2 = k + 1
                if k2 >= inst["N"]:
                    success = inst["gold_rank"] <= inst["N"]
                    r = rfn(success, inst["N"], inst["N"]) if reward_name == "bor" else rfn(success, inst["N"])
                    replay.append((sv, a, float(r), None, 1.0)); done = True
                else:
                    sv2 = state_vec(inst, k2)
                    replay.append((sv, a, -STEP_COST, sv2, 0.0)); k = k2
            step += 1
            if len(replay) >= BATCH:
                batch = random.sample(replay, BATCH)
                states = torch.tensor(np.stack([b[0] for b in batch]),
                                      dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch],
                                       dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch],
                                       dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch],
                                     dtype=torch.float32, device=device)
                nf = torch.tensor([b[3] is not None for b in batch],
                                  dtype=torch.bool, device=device)
                nq = torch.zeros(BATCH, dtype=torch.float32, device=device)
                if nf.any():
                    ns = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device)
                    with torch.no_grad(): nq[nf] = tgt(ns).max(1).values
                qv = net(states).gather(1, actions).squeeze(1)
                tv = rewards + (1-dones) * GAMMA * nq
                loss = nn.SmoothL1Loss()(qv, tv)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
            if step % TARGET_EVERY == 0:
                tgt.load_state_dict(net.state_dict())
        if ep % 2000 == 0:
            print(f"      {reward_name} {ep}/{TRAIN_EPS}", flush=True)
    return net

def rollout(inst, model):
    k = 1
    while True:
        sv = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
        if a == 0 or k >= inst["N"]: break
        k += 1
    return k

# ══════════════════════════════════════════════════════════════════════
# PART 4: Train 3 seeds
# ══════════════════════════════════════════════════════════════════════
print(f"[3/6] Training agents ({len(SEEDS)} seeds)...", flush=True)
t0 = time.time()
bor_models = {}
f1_models = {}
for seed in SEEDS:
    print(f"  Seed {seed}:", flush=True)
    bor_models[seed] = train_dqn(train_insts, "bor", seed)
    f1_models[seed] = train_dqn(train_insts, "f1", seed)
print(f"  All training done in {time.time()-t0:.0f}s")

# Sanity check (seed 0 only)
print(f"\n  BoR depth per bucket (seed={SEEDS[0]}):")
for bucket in ["1_easy", "2_medium", "3_hard", "4_vhard"]:
    sub = [inst for inst in test_insts if inst["bucket"] == bucket]
    if not sub: continue
    ks = [rollout(inst, bor_models[SEEDS[0]]) for inst in sub]
    fnd = [inst["gold_rank"] <= k for inst, k in zip(sub, ks)]
    print(f"    {bucket:>10}: K={np.mean(ks):.1f}, found={100*np.mean(fnd):.1f}%, n={len(sub)}")

# ══════════════════════════════════════════════════════════════════════
# PART 5: Claude API calls
# ══════════════════════════════════════════════════════════════════════
print("\n[4/6] Calling Claude API...", flush=True)

def call_claude_tool_use(query, tool_names_and_descs, retries=3):
    tool_names = [name for name, _ in tool_names_and_descs]
    tool_block = "\n".join(
        f"- {name}: {desc[:250]}"
        for name, desc in tool_names_and_descs
    )
    tools = [{
        "name": "select_tool",
        "description": "Select the most appropriate tool for the user query.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tool_name": {
                    "type": "string",
                    "enum": tool_names,
                    "description": "The name of the selected tool."
                }
            },
            "required": ["tool_name"]
        }
    }]
    user_msg = (
        f"Select the single most appropriate tool for this query.\n\n"
        f"Query: {query}\n\n"
        f"Available tools:\n{tool_block}"
    )
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=200,
                temperature=0,
                tools=tools,
                tool_choice={"type": "tool", "name": "select_tool"},
                messages=[{"role": "user", "content": user_msg}]
            )
            for block in resp.content:
                if block.type == "tool_use" and block.name == "select_tool":
                    return block.input.get("tool_name", "")
            return ""
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** (attempt + 1))
            else:
                print(f"    API error: {e}")
                return ""

# Cache for Claude calls: (query_hash, frozenset(tool_ids)) -> answer
claude_cache = {}
api_calls = 0

def cached_claude_call(query, tool_ids, inst):
    global api_calls
    cache_key = (hash(query), tuple(sorted(tool_ids)))
    if cache_key in claude_cache:
        return claude_cache[cache_key]
    tool_set = [(tool_names_list[tid], tool_descs_map[tool_names_list[tid]][:250])
                for tid in tool_ids]
    answer = call_claude_tool_use(query, tool_set)
    api_calls += 1
    claude_cache[cache_key] = answer
    return answer

# Run all seeds
all_seed_results = []   # list of (seed, method, bucket, gold_rank, k, presented, correct)
total = len(test_insts)

for qi, inst in enumerate(test_insts):
    if (qi+1) % 25 == 0 or qi == 0:
        print(f"  Query {qi+1}/{total} (API calls: {api_calls})", flush=True)

    gold = inst["gt_tool"]
    bucket = inst["bucket"]
    ranked_ids = inst["ranked_tool_ids"]

    # FK5 and FK1: deterministic, same across seeds. Call once, record for all seeds.
    for method, k in [("FK5", 5), ("FK1", 1)]:
        gold_presented = inst["gold_rank"] <= k
        if not gold_presented:
            correct = False
        else:
            answer = cached_claude_call(inst["query"], ranked_ids[:k], inst)
            correct = (answer == gold)
        for seed in SEEDS:
            all_seed_results.append({
                "seed": seed, "method": method, "bucket": bucket,
                "gold_rank": inst["gold_rank"], "k": k,
                "gold_presented": gold_presented, "claude_correct": correct
            })

    # BoR and F1: per-seed (K may differ)
    for seed in SEEDS:
        for method, model_dict in [("BoR", bor_models), ("F1", f1_models)]:
            k = min(rollout(inst, model_dict[seed]), MAX_TOOLS_IN_PROMPT)
            gold_presented = inst["gold_rank"] <= k
            if not gold_presented:
                correct = False
            else:
                answer = cached_claude_call(inst["query"], ranked_ids[:k], inst)
                correct = (answer == gold)
            all_seed_results.append({
                "seed": seed, "method": method, "bucket": bucket,
                "gold_rank": inst["gold_rank"], "k": k,
                "gold_presented": gold_presented, "claude_correct": correct
            })

print(f"  Done. Total API calls: {api_calls} (cache saved {len(claude_cache)} unique)")

# ══════════════════════════════════════════════════════════════════════
# PART 6: Report with std
# ══════════════════════════════════════════════════════════════════════
print("\n[5/6] Results")
print("=" * 80)
print(f"  DOWNSTREAM TOOL-CHOICE ACCURACY ({len(SEEDS)} seeds)")
print(f"  Model: {CLAUDE_MODEL}")
print(f"  BFCL, N={CAND_N}, {len(test_insts)} test queries")
print("=" * 80)

def compute_metrics(rows):
    pres = np.mean([r["gold_presented"] for r in rows])
    pres_rows = [r for r in rows if r["gold_presented"]]
    choice = np.mean([r["claude_correct"] for r in pres_rows]) if pres_rows else 0
    e2e = np.mean([r["claude_correct"] for r in rows])
    avg_k = np.mean([r["k"] for r in rows])
    return pres, choice, e2e, avg_k

# Overall with std
print(f"\n{'Method':>8} | {'Presented':>12} | {'Choice Acc':>14} | {'End-to-End':>14} | {'Avg K':>8}")
print("-" * 68)
for method in ["BoR", "F1", "FK5", "FK1"]:
    seed_metrics = []
    for seed in SEEDS:
        rows = [r for r in all_seed_results if r["method"] == method and r["seed"] == seed]
        if not rows: continue
        seed_metrics.append(compute_metrics(rows))
    if not seed_metrics: continue
    pres_vals = [m[0] for m in seed_metrics]
    choice_vals = [m[1] for m in seed_metrics]
    e2e_vals = [m[2] for m in seed_metrics]
    k_vals = [m[3] for m in seed_metrics]
    print(f"{method:>8} | "
          f"{100*np.mean(pres_vals):>5.1f}±{100*np.std(pres_vals):>4.1f}% | "
          f"{100*np.mean(choice_vals):>6.1f}±{100*np.std(choice_vals):>4.1f}% | "
          f"{100*np.mean(e2e_vals):>6.1f}±{100*np.std(e2e_vals):>4.1f}% | "
          f"{np.mean(k_vals):>4.1f}±{np.std(k_vals):.1f}")

# Per bucket with std
print(f"\nPer difficulty bucket (mean±std across {len(SEEDS)} seeds):")
print(f"{'Bucket':>10} | {'Method':>6} | {'n':>4} | {'Presented':>12} | "
      f"{'Choice%':>10} | {'E2E%':>10} | {'K':>8}")
print("-" * 75)
all_buckets = sorted(set(r["bucket"] for r in all_seed_results))
for bucket in all_buckets:
    for method in ["BoR", "F1", "FK5", "FK1"]:
        seed_metrics = []
        for seed in SEEDS:
            rows = [r for r in all_seed_results
                    if r["method"] == method and r["seed"] == seed and r["bucket"] == bucket]
            if not rows: continue
            seed_metrics.append(compute_metrics(rows))
        if not seed_metrics: continue
        n = len([r for r in all_seed_results
                 if r["method"] == method and r["seed"] == SEEDS[0] and r["bucket"] == bucket])
        pres_vals = [m[0] for m in seed_metrics]
        choice_vals = [m[1] for m in seed_metrics]
        e2e_vals = [m[2] for m in seed_metrics]
        k_vals = [m[3] for m in seed_metrics]
        print(f"{bucket:>10} | {method:>6} | {n:>4} | "
              f"{100*np.mean(pres_vals):>5.1f}±{100*np.std(pres_vals):>4.1f}% | "
              f"{100*np.mean(choice_vals):>5.1f}±{100*np.std(choice_vals):>3.1f}% | "
              f"{100*np.mean(e2e_vals):>5.1f}±{100*np.std(e2e_vals):>3.1f}% | "
              f"{np.mean(k_vals):>4.1f}±{np.std(k_vals):.1f}")
    print("-" * 75)

# Save everything
print("\n[6/6] Saving...", flush=True)
with open("downstream_results.json", "w") as f:
    json.dump(all_seed_results, f, indent=2, default=str)
print(f"Saved {len(all_seed_results)} results to downstream_results.json")
print("Done.")

[1/6] Loading BFCL data...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


BFCL_v3_simple.json: 0.00B [00:00, ?B/s]

  Loaded 400 entries
  Registry: 370 tools, Queries: 400
[2/6] Embedding...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  N=370, Train: 280, Test: 120
  Buckets: {'1_easy': 93, '2_medium': 24, '3_hard': 3}
[3/6] Training agents (3 seeds)...
  Seed 42:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  Seed 123:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  Seed 456:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  All training done in 180s

  BoR depth per bucket (seed=42):
        1_easy: K=1.4, found=100.0%, n=93
      2_medium: K=2.0, found=50.0%, n=24
        3_hard: K=1.7, found=0.0%, n=3

[4/6] Calling Claude API...
  Query 1/120 (API calls: 0)
  Query 25/120 (API calls: 50)
  Query 50/120 (API calls: 99)
  Query 75/120 (API calls: 162)
  Query 100/120 (API calls: 229)
  Done

In [2]:
# ══════════════════════════════════════════════════════════════════════
# BFCL Main-Table Numbers with 3-Seed Std
# ──────────────────────────────────────────────────────────────────────
# Reproduces the BFCL row from Table 1 (Section 4.1) with 3 seeds.
# Uses BM25 scorer (matches original notebook). Self-contained.
# Runtime: ~8 min | No API calls needed.
# ══════════════════════════════════════════════════════════════════════

SEEDS = [42, 123, 456]
TRAIN_EPS = 15000

import subprocess, sys, json, random, math, numpy as np
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "huggingface_hub", "rank_bm25", "torch"])

import torch, torch.nn as nn, torch.optim as optim
from collections import deque
from rank_bm25 import BM25Okapi
from huggingface_hub import hf_hub_download

# ── Load BFCL ──
print("[1/3] Loading BFCL...", flush=True)
fpath = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_simple.json",
    repo_type="dataset"
)
with open(fpath) as f:
    raw = [json.loads(line) for line in f]

registry = {}
entries = []
for entry in raw:
    question = entry.get('question', [])
    query_text = ''
    if isinstance(question, list) and question:
        inner = question[0]
        if isinstance(inner, list) and inner:
            for msg in inner:
                if isinstance(msg, dict) and msg.get('role') == 'user':
                    query_text = msg.get('content', '')
                    break
        elif isinstance(inner, dict):
            if inner.get('role') == 'user':
                query_text = inner.get('content', '')
    if not query_text: continue
    funcs = entry.get('function', [])
    if not funcs or not isinstance(funcs, list): continue
    func = funcs[0]
    name = func.get('name', '')
    desc = func.get('description', '')
    if not name: continue
    params = func.get('parameters', {})
    param_names = list(params.get('properties', {}).keys()) if isinstance(params, dict) else []
    text = f"{name}: {desc}. Parameters: {', '.join(param_names)}" if param_names else f"{name}: {desc}"
    registry[name] = text
    entries.append({'text': query_text[:500], 'correct_tool': name})

tool_names = list(registry.keys())
tool_descs = [registry[n] for n in tool_names]
tool_name_to_idx = {n: i for i, n in enumerate(tool_names)}
N = len(tool_names)

query_list = []
for e in entries:
    if e['correct_tool'] in tool_name_to_idx:
        query_list.append({
            'text': e['text'],
            'rel': {tool_name_to_idx[e['correct_tool']]},
            'R_q': 1,
            'correct_tool': e['correct_tool'],
        })

# BM25 scoring
bm25 = BM25Okapi([d.lower().split() for d in tool_descs])
for q in query_list:
    scores = bm25.get_scores(q['text'].lower().split())
    q['ranked'] = sorted(enumerate(scores), key=lambda x: -x[1])

print(f"  {N} tools, {len(query_list)} queries")

# ── Environment ──
class Env:
    def __init__(self, queries, N, dk=1, max_k=100):
        self.queries, self.N, self.dk, self.max_k = queries, N, dk, max_k
    def new_query(self):
        self.q = random.choice(self.queries)
        self.k = 0; self.found = False
        self.top_score = 0; self.gap = 0; self.score_std = 0
        return self._s()
    def step(self, a):
        if a == 0: return self._s(), self._r(), True
        self.k = min(self.k + self.dk, self.max_k)
        ranked = self.q['ranked'][:self.k]
        self.found = any(idx in self.q['rel'] for idx, _ in ranked)
        if ranked:
            sc = [s for _, s in ranked]
            self.top_score = sc[0]
            t3 = sc[:min(3, len(sc))]
            rest = sc[min(3, len(sc)):min(10, len(sc))]
            self.gap = np.mean(t3) - np.mean(rest) if rest else 0
            self.score_std = np.std(sc)
        if self.k >= self.max_k: return self._s(), self._r(), True
        return self._s(), None, False
    def _p_rand(self, k):
        R_q = self.q['R_q']
        if k >= self.N or R_q >= self.N: return 1.0
        return max(1e-12, 1 - math.comb(self.N - R_q, k) / math.comb(self.N, k))
    def _s(self):
        k = max(self.k, 1)
        lam = k * self.q['R_q'] / self.N
        p_rand = self._p_rand(k)
        bor_ceil = -math.log2(p_rand)
        return np.array([
            self.k / self.max_k,
            self.top_score / 15.0,
            self.gap / 5.0,
            self.score_std / 5.0,
            min(lam, 5.0) / 5.0,
            min(bor_ceil, 10.0) / 10.0,
            float(self.found),
        ], dtype=np.float32)
    def _r(self):
        k = max(self.k, 1)
        p = self._p_rand(k)
        return {'bor': -math.log2(p) if self.found else 0.0,
                'f1': 1.0 if self.found else 0.0,
                'p_rand': p, 'k': k, 'found': self.found, 'R_q': self.q['R_q']}

# ── DQN ──
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(7,64), nn.ReLU(), nn.Linear(64,64), nn.ReLU(), nn.Linear(64,2))
    def forward(self, x): return self.net(x)

class Buf:
    def __init__(self, cap=20000):
        self.b = deque(maxlen=cap)
    def push(self, *a): self.b.append(a)
    def sample(self, n):
        batch = random.sample(self.b, min(n, len(self.b)))
        s,a,r,s2,d = zip(*batch)
        return (torch.FloatTensor(np.array(s)), torch.LongTensor(a),
                torch.FloatTensor(r), torch.FloatTensor(np.array(s2)), torch.FloatTensor(d))
    def __len__(self): return len(self.b)

def train_dqn(env, rk, seed, n_eps=15000, sc=0.005, bs=64, gamma=0.95):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    pol=DQN(); tgt=DQN(); tgt.load_state_dict(pol.state_dict())
    opt=optim.Adam(pol.parameters(), lr=1e-3); buf=Buf(); eps=0.5
    for ep in range(n_eps):
        if ep>n_eps*0.7: eps=0.03
        elif ep>n_eps*0.4: eps=0.1
        s=env.new_query()
        while True:
            if random.random()<eps: a=random.randint(0,1)
            else:
                with torch.no_grad(): a=pol(torch.FloatTensor(s).unsqueeze(0)).argmax(1).item()
            s2,rw,done=env.step(a)
            r=rw[rk] if done else -sc
            buf.push(s,a,r,s2,float(done)); s=s2
            if len(buf)>=bs:
                sb,ab,rb,s2b,db=buf.sample(bs)
                qc=pol(sb).gather(1,ab.unsqueeze(1)).squeeze(1)
                with torch.no_grad(): qt=rb+gamma*tgt(s2b).max(1)[0]*(1-db)
                loss=nn.MSELoss()(qc,qt); opt.zero_grad(); loss.backward(); opt.step()
            if done: break
        if ep%500==0: tgt.load_state_dict(pol.state_dict())
        if (ep+1)%5000==0: print(f"{ep+1}/{n_eps}", end=" ", flush=True)
    return pol

def evaluate(env, pol, qs, runs=5):
    res=[]
    for _ in range(runs):
        for q in qs:
            env.queries=[q]; s=env.new_query()
            while True:
                with torch.no_grad(): a=pol(torch.FloatTensor(s).unsqueeze(0)).argmax(1).item()
                s2,rw,done=env.step(a)
                if done: res.append(rw); break
                s=s2
    return res

def eval_fixed(env, qs, target_k):
    res=[]
    for q in qs:
        env.queries=[q]; env.new_query()
        for _ in range(target_k // env.dk): env.step(1)
        _,rw,_=env.step(0); res.append(rw)
    return res

def metrics(res):
    k = np.mean([r['k'] for r in res])
    k_std = np.std([r['k'] for r in res])
    found = np.mean([r['found'] for r in res]) * 100
    bor = np.mean([r['bor'] for r in res])
    return k, k_std, found, bor

# ── Train/test split (fixed across seeds) ──
random.seed(SEEDS[0])
random.shuffle(query_list)
split = int(len(query_list) * 0.7)
train_q, test_q = query_list[:split], query_list[split:]
print(f"  Train: {len(train_q)}, Test: {len(test_q)}")

max_k = min(N, 100)

# ── Fixed-K baselines (deterministic, run once) ──
print("\n[2/3] Fixed-K baselines...", flush=True)
te_env = Env(test_q, N, dk=1, max_k=max_k)
fk_results = {}
for fk in [1, 3, 5, 10, 20, 50]:
    if fk > N: break
    res = eval_fixed(te_env, test_q, fk)
    k, k_std, found, bor = metrics(res)
    fk_results[fk] = (k, k_std, found, bor)
    print(f"  FK={fk:>2}: Found={found:>5.1f}%  BoR={bor:>5.2f}")

# ── 3-seed training ──
print(f"\n[3/3] Training BoR + F1 ({len(SEEDS)} seeds)...", flush=True)
seed_bor = []
seed_f1 = []

for seed in SEEDS:
    print(f"\n  Seed {seed}:", flush=True)
    tr_env = Env(train_q, N, dk=1, max_k=max_k)
    te_env = Env(test_q, N, dk=1, max_k=max_k)

    print("    BoR: ", end="", flush=True)
    bor_pol = train_dqn(tr_env, 'bor', seed, n_eps=TRAIN_EPS)
    print("done", flush=True)

    print("    F1:  ", end="", flush=True)
    f1_pol = train_dqn(tr_env, 'f1', seed, n_eps=TRAIN_EPS)
    print("done", flush=True)

    br = evaluate(te_env, bor_pol, test_q)
    fr = evaluate(te_env, f1_pol, test_q)
    seed_bor.append(metrics(br))
    seed_f1.append(metrics(fr))

# ── Report ──
print("\n" + "=" * 70)
print(f"  BFCL MAIN TABLE: {N} tools, {len(test_q)} test queries, {len(SEEDS)} seeds")
print("=" * 70)

print(f"\n{'Method':<20} | {'K':>10} | {'K std':>8} | {'Found%':>12} | {'BoR bits':>12}")
print("-" * 70)

# BoR
bk = [s[0] for s in seed_bor]; bks = [s[1] for s in seed_bor]
bf = [s[2] for s in seed_bor]; bb = [s[3] for s in seed_bor]
print(f"{'BoR DQN':<20} | {np.mean(bk):>5.1f}±{np.std(bk):.1f} | "
      f"{np.mean(bks):>5.2f}±{np.std(bks):.2f} | "
      f"{np.mean(bf):>5.1f}±{np.std(bf):.1f} | "
      f"{np.mean(bb):>5.2f}±{np.std(bb):.2f}")

# F1
fk = [s[0] for s in seed_f1]; fks = [s[1] for s in seed_f1]
ff = [s[2] for s in seed_f1]; fb = [s[3] for s in seed_f1]
print(f"{'F1 DQN':<20} | {np.mean(fk):>5.1f}±{np.std(fk):.1f} | "
      f"{np.mean(fks):>5.2f}±{np.std(fks):.2f} | "
      f"{np.mean(ff):>5.1f}±{np.std(ff):.1f} | "
      f"{np.mean(fb):>5.2f}±{np.std(fb):.2f}")

# Fixed-K
for fk_val, (k, k_std, found, bor) in sorted(fk_results.items()):
    print(f"{'FK=' + str(fk_val):<20} | {k:>5.1f}     | {k_std:>5.2f}     | "
          f"{found:>5.1f}      | {bor:>5.2f}")

print("-" * 70)
print("\nValues after ± are std across 3 seeds. FK baselines are deterministic.")
print("Done.")

[1/3] Loading BFCL...
  370 tools, 400 queries
  Train: 280, Test: 120

[2/3] Fixed-K baselines...
  FK= 1: Found= 60.0%  BoR= 5.12
  FK= 3: Found= 78.3%  BoR= 5.44
  FK= 5: Found= 82.5%  BoR= 5.12
  FK=10: Found= 85.0%  BoR= 4.43
  FK=20: Found= 87.5%  BoR= 3.68
  FK=50: Found= 90.8%  BoR= 2.62

[3/3] Training BoR + F1 (3 seeds)...

  Seed 42:
    BoR: 5000/15000 10000/15000 15000/15000 done
    F1:  5000/15000 10000/15000 15000/15000 done

  Seed 123:
    BoR: 5000/15000 10000/15000 15000/15000 done
    F1:  5000/15000 10000/15000 15000/15000 done

  Seed 456:
    BoR: 5000/15000 10000/15000 15000/15000 done
    F1:  5000/15000 10000/15000 15000/15000 done

  BFCL MAIN TABLE: 370 tools, 120 test queries, 3 seeds

Method               |          K |    K std |       Found% |     BoR bits
----------------------------------------------------------------------
BoR DQN              |   7.4±2.5 | 17.21±6.53 |  90.3±2.4 |  7.08±0.09
F1 DQN               |   6.4±1.9 | 15.10±5.33 |  88.9±1.4 

In [4]:
# ══════════════════════════════════════════════════════════════════════
# Downstream Tool-Choice Validation: BoR vs Fixed-K via Claude API
# ──────────────────────────────────────────────────────────────────────
# BFCL with BM25 scorer (matches Section 4.1 main table).
# 3-seed training with std. Full 370-function registry.
#
# Cost: ~$3-5 | Runtime: ~15 min training + ~15 min API calls
# ══════════════════════════════════════════════════════════════════════

ANTHROPIC_API_KEY = "key"

# ── Config ──
SEEDS = [42, 123, 456]
CAND_N = 370               # full BFCL registry (matches Section 4.1)
MAX_QUERIES = 0            # 0 = all
TRAIN_EPS = 8000
CLAUDE_MODEL = "claude-sonnet-4-6"
MAX_TOOLS_IN_PROMPT = 30

# ── Installs ──
import sys, subprocess, importlib.util
def ensure(name, pip=None):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or name])
ensure("anthropic"); ensure("numpy"); ensure("torch"); ensure("rank_bm25")
ensure("sklearn", "scikit-learn")
ensure("huggingface_hub")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "sympy>=1.13.1"])

import json, math, random, time
from collections import deque, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from rank_bm25 import BM25Okapi
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download
import anthropic

device = "cuda" if torch.cuda.is_available() else "cpu"
assert ANTHROPIC_API_KEY.strip(), "Set ANTHROPIC_API_KEY before running."
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY.strip())

from huggingface_hub import login
login(token="YOUR_HF_TOKEN", add_to_git_credential=False)

# ══════════════════════════════════════════════════════════════════════
# PART 1: Load BFCL data
# ══════════════════════════════════════════════════════════════════════
print("[1/6] Loading BFCL data...", flush=True)
fpath = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_simple.json",
    repo_type="dataset"
)
with open(fpath) as f:
    raw = [json.loads(line) for line in f]
print(f"  Loaded {len(raw)} entries")

registry = {}
entries = []
for entry in raw:
    question = entry.get('question', [])
    query_text = ''
    if isinstance(question, list) and question:
        inner = question[0]
        if isinstance(inner, list) and inner:
            for msg in inner:
                if isinstance(msg, dict) and msg.get('role') == 'user':
                    query_text = msg.get('content', '')
                    break
        elif isinstance(inner, dict):
            if inner.get('role') == 'user':
                query_text = inner.get('content', '')
    if not query_text: continue
    funcs = entry.get('function', [])
    if not funcs or not isinstance(funcs, list): continue
    func = funcs[0]
    name = func.get('name', '')
    desc = func.get('description', '')
    if not name: continue
    params = func.get('parameters', {})
    param_names = list(params.get('properties', {}).keys()) if isinstance(params, dict) else []
    text = f"{name}: {desc}. Parameters: {', '.join(param_names)}" if param_names else f"{name}: {desc}"
    registry[name] = text
    entries.append({'query': query_text[:500], 'gt_tool': name})

tool_names_list = sorted(registry.keys())
tool_descs_list = [registry[n] for n in tool_names_list]
tool_descs_map = {n: registry[n] for n in tool_names_list}
tool_to_idx = {n: i for i, n in enumerate(tool_names_list)}
N_FULL = len(tool_names_list)

query_list = [e for e in entries if e['gt_tool'] in tool_to_idx]
random.seed(SEEDS[0]); np.random.seed(SEEDS[0]); torch.manual_seed(SEEDS[0])
if MAX_QUERIES > 0 and len(query_list) > MAX_QUERIES:
    random.shuffle(query_list)
    query_list = query_list[:MAX_QUERIES]
print(f"  Registry: {N_FULL} tools, Queries: {len(query_list)}")
assert len(query_list) > 0, "No queries loaded."

# ══════════════════════════════════════════════════════════════════════
# PART 2: BM25 scoring (matches Section 4.1)
# ══════════════════════════════════════════════════════════════════════
print("[2/6] BM25 scoring...", flush=True)
bm25 = BM25Okapi([d.lower().split() for d in tool_descs_list])
score_matrix = np.zeros((len(query_list), N_FULL), dtype=np.float32)
for i, q in enumerate(query_list):
    score_matrix[i] = bm25.get_scores(q["query"].lower().split())

# Quick check: found@1 should be ~60% (matching Section 4.1)
found_at_1 = np.mean([
    tool_to_idx[q["gt_tool"]] == np.argmax(score_matrix[i])
    for i, q in enumerate(query_list)
])
print(f"  BM25 found@1: {100*found_at_1:.1f}%")

def difficulty_bucket(rank):
    if rank == 1: return "1_easy"
    if 2 <= rank <= 5: return "2_medium"
    if 6 <= rank <= 20: return "3_hard"
    return "4_vhard"

base_instances = []
for i, q in enumerate(query_list):
    gold_idx = tool_to_idx[q["gt_tool"]]
    ranked_global = np.argsort(-score_matrix[i])
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({"query": q["query"], "gold_idx": gold_idx,
        "scores_all": score_matrix[i], "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full, "gt_tool": q["gt_tool"]})

indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEEDS[0])

CAND_N = min(CAND_N, N_FULL)

def make_cand(base, N):
    gold_idx = base["gold_idx"]
    hard = [j for j in base["ranked_global"] if j != gold_idx][:N-1]
    cand = [gold_idx] + list(hard)
    scores = np.array([base["scores_all"][j] for j in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_ids = [cand[j] for j in order]
    ranked_sc = scores[order]
    gold_rank = int(ranked_ids.index(gold_idx) + 1)
    return {"gold_rank": gold_rank, "ranked_tool_ids": ranked_ids,
            "scores": ranked_sc, "N": N, "query": base["query"],
            "bucket": difficulty_bucket(gold_rank), "gt_tool": base["gt_tool"]}

instances = [make_cand(base_instances[i], CAND_N) for i in range(len(base_instances))]
train_insts = [instances[i] for i in train_idx]
test_insts = [instances[i] for i in test_idx]
print(f"  N={CAND_N}, Train: {len(train_insts)}, Test: {len(test_insts)}")
buckets = defaultdict(int)
for inst in test_insts: buckets[inst["bucket"]] += 1
print(f"  Buckets: {dict(sorted(buckets.items()))}")

# ══════════════════════════════════════════════════════════════════════
# PART 3: DQN infrastructure
# ══════════════════════════════════════════════════════════════════════
BATCH = 128; REPLAY_SZ = 50000; LR = 1e-3; GAMMA = 0.95; STEP_COST = 0.01
TARGET_EVERY = 500

def bor_reward(success, k, N):
    return -math.log2(max(k / N, 1e-12)) if success else 0.0

def f1_reward(success, k):
    return (2.0 / (k + 1.0)) if success else 0.0

def state_vec(inst, k):
    s = inst["scores"]; N = inst["N"]; idx = min(k-1, N-1)
    cur = float(s[idx])
    nxt = float(s[idx+1]) if idx+1 < N else float(s[idx])
    first = float(s[0]); mean = float(s.mean()); std = float(s.std() + 1e-6)
    gap = cur - nxt if idx+1 < N else 0.0
    return np.array([k/N, math.log2(k+1)/math.log2(N+1), cur, nxt, gap,
                     (cur-mean)/std, cur/(abs(first)+1e-6)], dtype=np.float32)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7,64), nn.ReLU(),
            nn.Linear(64,64), nn.ReLU(),
            nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def train_dqn(train_insts, reward_name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rfn = bor_reward if reward_name == "bor" else f1_reward
    net = QNet().to(device); tgt = QNet().to(device)
    tgt.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SZ)
    eps_s, eps_e = 1.0, 0.05; step = 0
    for ep in range(1, TRAIN_EPS+1):
        inst = random.choice(train_insts); k = 1; done = False
        eps = eps_e + (eps_s - eps_e) * max(0, 1 - ep/TRAIN_EPS)
        while not done:
            sv = state_vec(inst, k)
            if random.random() < eps: a = random.randint(0,1)
            else:
                with torch.no_grad():
                    a = int(net(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
            if k >= inst["N"]: a = 0
            if a == 0:
                success = inst["gold_rank"] <= k
                r = rfn(success, k, inst["N"]) if reward_name == "bor" else rfn(success, k)
                replay.append((sv, a, float(r), None, 1.0)); done = True
            else:
                k2 = k + 1
                if k2 >= inst["N"]:
                    success = inst["gold_rank"] <= inst["N"]
                    r = rfn(success, inst["N"], inst["N"]) if reward_name == "bor" else rfn(success, inst["N"])
                    replay.append((sv, a, float(r), None, 1.0)); done = True
                else:
                    sv2 = state_vec(inst, k2)
                    replay.append((sv, a, -STEP_COST, sv2, 0.0)); k = k2
            step += 1
            if len(replay) >= BATCH:
                batch = random.sample(replay, BATCH)
                states = torch.tensor(np.stack([b[0] for b in batch]),
                                      dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch],
                                       dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch],
                                       dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch],
                                     dtype=torch.float32, device=device)
                nf = torch.tensor([b[3] is not None for b in batch],
                                  dtype=torch.bool, device=device)
                nq = torch.zeros(BATCH, dtype=torch.float32, device=device)
                if nf.any():
                    ns = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device)
                    with torch.no_grad(): nq[nf] = tgt(ns).max(1).values
                qv = net(states).gather(1, actions).squeeze(1)
                tv = rewards + (1-dones) * GAMMA * nq
                loss = nn.SmoothL1Loss()(qv, tv)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
            if step % TARGET_EVERY == 0:
                tgt.load_state_dict(net.state_dict())
        if ep % 2000 == 0:
            print(f"      {reward_name} {ep}/{TRAIN_EPS}", flush=True)
    return net

def rollout(inst, model):
    k = 1
    while True:
        sv = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
        if a == 0 or k >= inst["N"]: break
        k += 1
    return k

# ══════════════════════════════════════════════════════════════════════
# PART 4: Train 3 seeds
# ══════════════════════════════════════════════════════════════════════
print(f"[3/6] Training agents ({len(SEEDS)} seeds)...", flush=True)
t0 = time.time()
bor_models = {}
f1_models = {}
for seed in SEEDS:
    print(f"  Seed {seed}:", flush=True)
    bor_models[seed] = train_dqn(train_insts, "bor", seed)
    f1_models[seed] = train_dqn(train_insts, "f1", seed)
print(f"  All training done in {time.time()-t0:.0f}s")

# Sanity check
print(f"\n  BoR depth per bucket (seed={SEEDS[0]}):")
for bucket in ["1_easy", "2_medium", "3_hard", "4_vhard"]:
    sub = [inst for inst in test_insts if inst["bucket"] == bucket]
    if not sub: continue
    ks = [rollout(inst, bor_models[SEEDS[0]]) for inst in sub]
    fnd = [inst["gold_rank"] <= k for inst, k in zip(sub, ks)]
    print(f"    {bucket:>10}: K={np.mean(ks):.1f}, found={100*np.mean(fnd):.1f}%, n={len(sub)}")

# ══════════════════════════════════════════════════════════════════════
# PART 5: Claude API calls
# ══════════════════════════════════════════════════════════════════════
print("\n[4/6] Calling Claude API...", flush=True)

def call_claude_tool_use(query, tool_names_and_descs, retries=3):
    tool_names = [name for name, _ in tool_names_and_descs]
    tool_block = "\n".join(
        f"- {name}: {desc[:250]}"
        for name, desc in tool_names_and_descs
    )
    tools = [{
        "name": "select_tool",
        "description": "Select the most appropriate tool for the user query.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tool_name": {
                    "type": "string",
                    "enum": tool_names,
                    "description": "The name of the selected tool."
                }
            },
            "required": ["tool_name"]
        }
    }]
    user_msg = (
        f"Select the single most appropriate tool for this query.\n\n"
        f"Query: {query}\n\n"
        f"Available tools:\n{tool_block}"
    )
    for attempt in range(retries):
        try:
            resp = client.messages.create(
                model=CLAUDE_MODEL, max_tokens=200,
                temperature=0,
                tools=tools,
                tool_choice={"type": "tool", "name": "select_tool"},
                messages=[{"role": "user", "content": user_msg}]
            )
            for block in resp.content:
                if block.type == "tool_use" and block.name == "select_tool":
                    return block.input.get("tool_name", "")
            return ""
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** (attempt + 1))
            else:
                print(f"    API error: {e}")
                return ""

# Cache: (query_hash, tuple(tool_ids)) -> answer
claude_cache = {}
api_calls = 0

def cached_claude_call(query, tool_ids):
    global api_calls
    cache_key = (hash(query), tuple(sorted(tool_ids)))
    if cache_key in claude_cache:
        return claude_cache[cache_key]
    tool_set = [(tool_names_list[tid], tool_descs_map[tool_names_list[tid]][:250])
                for tid in tool_ids]
    answer = call_claude_tool_use(query, tool_set)
    api_calls += 1
    claude_cache[cache_key] = answer
    return answer

all_seed_results = []
total = len(test_insts)

for qi, inst in enumerate(test_insts):
    if (qi+1) % 25 == 0 or qi == 0:
        print(f"  Query {qi+1}/{total} (API calls: {api_calls})", flush=True)

    gold = inst["gt_tool"]
    bucket = inst["bucket"]
    ranked_ids = inst["ranked_tool_ids"]

    # FK5 and FK1: deterministic
    for method, k in [("FK5", 5), ("FK1", 1)]:
        gold_presented = inst["gold_rank"] <= k
        if not gold_presented:
            correct = False
        else:
            answer = cached_claude_call(inst["query"], ranked_ids[:k])
            correct = (answer == gold)
        for seed in SEEDS:
            all_seed_results.append({
                "seed": seed, "method": method, "bucket": bucket,
                "gold_rank": inst["gold_rank"], "k": k,
                "gold_presented": gold_presented, "claude_correct": correct
            })

    # BoR and F1: per-seed
    for seed in SEEDS:
        for method, model_dict in [("BoR", bor_models), ("F1", f1_models)]:
            k = min(rollout(inst, model_dict[seed]), MAX_TOOLS_IN_PROMPT)
            gold_presented = inst["gold_rank"] <= k
            if not gold_presented:
                correct = False
            else:
                answer = cached_claude_call(inst["query"], ranked_ids[:k])
                correct = (answer == gold)
            all_seed_results.append({
                "seed": seed, "method": method, "bucket": bucket,
                "gold_rank": inst["gold_rank"], "k": k,
                "gold_presented": gold_presented, "claude_correct": correct
            })

print(f"  Done. Total API calls: {api_calls} (cache saved {len(claude_cache)} unique)")

# ══════════════════════════════════════════════════════════════════════
# PART 6: Report
# ══════════════════════════════════════════════════════════════════════
print("\n[5/6] Results")
print("=" * 80)
print(f"  DOWNSTREAM TOOL-CHOICE ACCURACY ({len(SEEDS)} seeds, BM25 scorer)")
print(f"  Model: {CLAUDE_MODEL}")
print(f"  BFCL, N={CAND_N}, {len(test_insts)} test queries")
print("=" * 80)

def compute_metrics(rows):
    pres = np.mean([r["gold_presented"] for r in rows])
    pres_rows = [r for r in rows if r["gold_presented"]]
    choice = np.mean([r["claude_correct"] for r in pres_rows]) if pres_rows else 0
    e2e = np.mean([r["claude_correct"] for r in rows])
    avg_k = np.mean([r["k"] for r in rows])
    return pres, choice, e2e, avg_k

print(f"\n{'Method':>8} | {'Presented':>12} | {'Choice Acc':>14} | {'End-to-End':>14} | {'Avg K':>8}")
print("-" * 68)
for method in ["BoR", "F1", "FK5", "FK1"]:
    seed_metrics = []
    for seed in SEEDS:
        rows = [r for r in all_seed_results if r["method"] == method and r["seed"] == seed]
        if not rows: continue
        seed_metrics.append(compute_metrics(rows))
    if not seed_metrics: continue
    pres_vals = [m[0] for m in seed_metrics]
    choice_vals = [m[1] for m in seed_metrics]
    e2e_vals = [m[2] for m in seed_metrics]
    k_vals = [m[3] for m in seed_metrics]
    print(f"{method:>8} | "
          f"{100*np.mean(pres_vals):>5.1f}±{100*np.std(pres_vals):>4.1f}% | "
          f"{100*np.mean(choice_vals):>6.1f}±{100*np.std(choice_vals):>4.1f}% | "
          f"{100*np.mean(e2e_vals):>6.1f}±{100*np.std(e2e_vals):>4.1f}% | "
          f"{np.mean(k_vals):>4.1f}±{np.std(k_vals):.1f}")

print(f"\nPer difficulty bucket (mean±std across {len(SEEDS)} seeds):")
print(f"{'Bucket':>10} | {'Method':>6} | {'n':>4} | {'Presented':>12} | "
      f"{'Choice%':>10} | {'E2E%':>10} | {'K':>8}")
print("-" * 75)
all_buckets = sorted(set(r["bucket"] for r in all_seed_results))
for bucket in all_buckets:
    for method in ["BoR", "F1", "FK5", "FK1"]:
        seed_metrics = []
        for seed in SEEDS:
            rows = [r for r in all_seed_results
                    if r["method"] == method and r["seed"] == seed and r["bucket"] == bucket]
            if not rows: continue
            seed_metrics.append(compute_metrics(rows))
        if not seed_metrics: continue
        n = len([r for r in all_seed_results
                 if r["method"] == method and r["seed"] == SEEDS[0] and r["bucket"] == bucket])
        pres_vals = [m[0] for m in seed_metrics]
        choice_vals = [m[1] for m in seed_metrics]
        e2e_vals = [m[2] for m in seed_metrics]
        k_vals = [m[3] for m in seed_metrics]
        print(f"{bucket:>10} | {method:>6} | {n:>4} | "
              f"{100*np.mean(pres_vals):>5.1f}±{100*np.std(pres_vals):>4.1f}% | "
              f"{100*np.mean(choice_vals):>5.1f}±{100*np.std(choice_vals):>3.1f}% | "
              f"{100*np.mean(e2e_vals):>5.1f}±{100*np.std(e2e_vals):>3.1f}% | "
              f"{np.mean(k_vals):>4.1f}±{np.std(k_vals):.1f}")
    print("-" * 75)

print("\n[6/6] Saving...", flush=True)
with open("downstream_results_bm25.json", "w") as f:
    json.dump(all_seed_results, f, indent=2, default=str)
print(f"Saved {len(all_seed_results)} results to downstream_results_bm25.json")
print("Done.")

[1/6] Loading BFCL data...
  Loaded 400 entries
  Registry: 370 tools, Queries: 400
[2/6] BM25 scoring...
  BM25 found@1: 65.0%
  N=370, Train: 280, Test: 120
  Buckets: {'1_easy': 78, '2_medium': 23, '3_hard': 7, '4_vhard': 12}
[3/6] Training agents (3 seeds)...
  Seed 42:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  Seed 123:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  Seed 456:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000
  All training done in 232s

  BoR depth per bucket (seed=42):
        1_easy: K=1.4, found=100.0%, n=78
      2_medium: K=2.0, found=60.9%, n=23
        3_hard: K=2.0, found=0.0%, n=7
       4_vhard: K=1.9, found=0.0%, n=12

[4/6] C

In [5]:
# ══════════════════════════════════════════════════════════════════════
# BFCL Embedding Scorer: Main-Table Numbers with 3-Seed Std
# ──────────────────────────────────────────────────────────────────────
# MiniLM-L6-v2 embeddings. Matches the embedding downstream experiment.
# No API calls. Runtime: ~10 min.
# ══════════════════════════════════════════════════════════════════════

SEEDS = [42, 123, 456]
CAND_N = 370               # full registry
TRAIN_EPS = 8000

import sys, subprocess, importlib.util
def ensure(name, pip=None):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or name])
ensure("numpy"); ensure("torch")
ensure("sentence_transformers", "sentence-transformers")
ensure("sklearn", "scikit-learn")
ensure("huggingface_hub")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "sympy>=1.13.1"])

import json, math, random, time
from collections import deque, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from huggingface_hub import hf_hub_download

from huggingface_hub import login
login(token="YOUR_HF_TOKEN", add_to_git_credential=False)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load BFCL ──
print("[1/3] Loading BFCL...", flush=True)
fpath = hf_hub_download(
    repo_id="gorilla-llm/Berkeley-Function-Calling-Leaderboard",
    filename="BFCL_v3_simple.json",
    repo_type="dataset"
)
with open(fpath) as f:
    raw = [json.loads(line) for line in f]

registry = {}
entries = []
for entry in raw:
    question = entry.get('question', [])
    query_text = ''
    if isinstance(question, list) and question:
        inner = question[0]
        if isinstance(inner, list) and inner:
            for msg in inner:
                if isinstance(msg, dict) and msg.get('role') == 'user':
                    query_text = msg.get('content', '')
                    break
        elif isinstance(inner, dict):
            if inner.get('role') == 'user':
                query_text = inner.get('content', '')
    if not query_text: continue
    funcs = entry.get('function', [])
    if not funcs or not isinstance(funcs, list): continue
    func = funcs[0]
    name = func.get('name', '')
    desc = func.get('description', '')
    if not name: continue
    params = func.get('parameters', {})
    param_names = list(params.get('properties', {}).keys()) if isinstance(params, dict) else []
    text = f"{name}: {desc}. Parameters: {', '.join(param_names)}" if param_names else f"{name}: {desc}"
    registry[name] = text
    entries.append({'query': query_text[:500], 'gt_tool': name})

tool_names_list = sorted(registry.keys())
tool_descs_list = [registry[n] for n in tool_names_list]
tool_to_idx = {n: i for i, n in enumerate(tool_names_list)}
N_FULL = len(tool_names_list)

query_list = [e for e in entries if e['gt_tool'] in tool_to_idx]
random.seed(SEEDS[0]); np.random.seed(SEEDS[0])
print(f"  {N_FULL} tools, {len(query_list)} queries")

# ── Embedding scoring ──
print("[2/3] Embedding...", flush=True)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tool_embs = encoder.encode(tool_descs_list, batch_size=64,
                           show_progress_bar=False, normalize_embeddings=True)
q_embs = encoder.encode([q["query"] for q in query_list], batch_size=64,
                        show_progress_bar=False, normalize_embeddings=True)
score_matrix = q_embs @ tool_embs.T

found_at_1 = np.mean([
    tool_to_idx[q["gt_tool"]] == np.argmax(score_matrix[i])
    for i, q in enumerate(query_list)
])
print(f"  Embed found@1: {100*found_at_1:.1f}%")

def difficulty_bucket(rank):
    if rank == 1: return "1_easy"
    if 2 <= rank <= 5: return "2_medium"
    if 6 <= rank <= 20: return "3_hard"
    return "4_vhard"

base_instances = []
for i, q in enumerate(query_list):
    gold_idx = tool_to_idx[q["gt_tool"]]
    ranked_global = np.argsort(-score_matrix[i])
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({"query": q["query"], "gold_idx": gold_idx,
        "scores_all": score_matrix[i], "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full, "gt_tool": q["gt_tool"]})

indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEEDS[0])

CAND_N = min(CAND_N, N_FULL)

def make_cand(base, N):
    gold_idx = base["gold_idx"]
    hard = [j for j in base["ranked_global"] if j != gold_idx][:N-1]
    cand = [gold_idx] + list(hard)
    scores = np.array([base["scores_all"][j] for j in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_ids = [cand[j] for j in order]
    ranked_sc = scores[order]
    gold_rank = int(ranked_ids.index(gold_idx) + 1)
    return {"gold_rank": gold_rank, "ranked_tool_ids": ranked_ids,
            "scores": ranked_sc, "N": N, "query": base["query"],
            "bucket": difficulty_bucket(gold_rank), "gt_tool": base["gt_tool"]}

instances = [make_cand(base_instances[i], CAND_N) for i in range(len(base_instances))]
train_insts = [instances[i] for i in train_idx]
test_insts = [instances[i] for i in test_idx]
print(f"  N={CAND_N}, Train: {len(train_insts)}, Test: {len(test_insts)}")
buckets = defaultdict(int)
for inst in test_insts: buckets[inst["bucket"]] += 1
print(f"  Buckets: {dict(sorted(buckets.items()))}")

# ── DQN ──
BATCH = 128; REPLAY_SZ = 50000; LR = 1e-3; GAMMA = 0.95; STEP_COST = 0.01
TARGET_EVERY = 500

def bor_reward(success, k, N):
    return -math.log2(max(k / N, 1e-12)) if success else 0.0

def f1_reward(success, k):
    return (2.0 / (k + 1.0)) if success else 0.0

def state_vec(inst, k):
    s = inst["scores"]; N = inst["N"]; idx = min(k-1, N-1)
    cur = float(s[idx])
    nxt = float(s[idx+1]) if idx+1 < N else float(s[idx])
    first = float(s[0]); mean = float(s.mean()); std = float(s.std() + 1e-6)
    gap = cur - nxt if idx+1 < N else 0.0
    return np.array([k/N, math.log2(k+1)/math.log2(N+1), cur, nxt, gap,
                     (cur-mean)/std, cur/(abs(first)+1e-6)], dtype=np.float32)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7,64), nn.ReLU(),
            nn.Linear(64,64), nn.ReLU(),
            nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def train_dqn(train_insts, reward_name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rfn = bor_reward if reward_name == "bor" else f1_reward
    net = QNet().to(device); tgt = QNet().to(device)
    tgt.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SZ)
    eps_s, eps_e = 1.0, 0.05; step = 0
    for ep in range(1, TRAIN_EPS+1):
        inst = random.choice(train_insts); k = 1; done = False
        eps = eps_e + (eps_s - eps_e) * max(0, 1 - ep/TRAIN_EPS)
        while not done:
            sv = state_vec(inst, k)
            if random.random() < eps: a = random.randint(0,1)
            else:
                with torch.no_grad():
                    a = int(net(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
            if k >= inst["N"]: a = 0
            if a == 0:
                success = inst["gold_rank"] <= k
                r = rfn(success, k, inst["N"]) if reward_name == "bor" else rfn(success, k)
                replay.append((sv, a, float(r), None, 1.0)); done = True
            else:
                k2 = k + 1
                if k2 >= inst["N"]:
                    success = inst["gold_rank"] <= inst["N"]
                    r = rfn(success, inst["N"], inst["N"]) if reward_name == "bor" else rfn(success, inst["N"])
                    replay.append((sv, a, float(r), None, 1.0)); done = True
                else:
                    sv2 = state_vec(inst, k2)
                    replay.append((sv, a, -STEP_COST, sv2, 0.0)); k = k2
            step += 1
            if len(replay) >= BATCH:
                batch = random.sample(replay, BATCH)
                states = torch.tensor(np.stack([b[0] for b in batch]),
                                      dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch],
                                       dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch],
                                       dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch],
                                     dtype=torch.float32, device=device)
                nf = torch.tensor([b[3] is not None for b in batch],
                                  dtype=torch.bool, device=device)
                nq = torch.zeros(BATCH, dtype=torch.float32, device=device)
                if nf.any():
                    ns = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device)
                    with torch.no_grad(): nq[nf] = tgt(ns).max(1).values
                qv = net(states).gather(1, actions).squeeze(1)
                tv = rewards + (1-dones) * GAMMA * nq
                loss = nn.SmoothL1Loss()(qv, tv)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
            if step % TARGET_EVERY == 0:
                tgt.load_state_dict(net.state_dict())
        if ep % 2000 == 0:
            print(f"      {reward_name} {ep}/{TRAIN_EPS}", flush=True)
    return net

def rollout(inst, model):
    k = 1
    while True:
        sv = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
        if a == 0 or k >= inst["N"]: break
        k += 1
    return k

# ── Fixed-K baselines ──
print("\n  Fixed-K baselines:")
fk_results = {}
for fk_val in [1, 3, 5, 10, 20, 50]:
    if fk_val > CAND_N: break
    found = np.mean([inst["gold_rank"] <= fk_val for inst in test_insts]) * 100
    bor_vals = [max(0, -math.log2(max(fk_val/inst["N"], 1e-12)))
                if inst["gold_rank"] <= fk_val else 0.0 for inst in test_insts]
    bor = np.mean(bor_vals)
    fk_results[fk_val] = (fk_val, 0, found, bor)
    print(f"    FK={fk_val:>2}: Found={found:>5.1f}%  BoR={bor:>5.2f}")

# ── 3-seed training ──
print(f"\n[3/3] Training BoR + F1 ({len(SEEDS)} seeds)...", flush=True)
t0 = time.time()
seed_bor = []
seed_f1 = []
seed_bor_buckets = []

for seed in SEEDS:
    print(f"\n  Seed {seed}:", flush=True)
    bor_model = train_dqn(train_insts, "bor", seed)
    f1_model = train_dqn(train_insts, "f1", seed)

    bor_ks = [rollout(inst, bor_model) for inst in test_insts]
    bor_found = [inst["gold_rank"] <= k for inst, k in zip(test_insts, bor_ks)]
    bor_bits = [max(0, -math.log2(max(k/inst["N"], 1e-12))) if f else 0.0
                for inst, k, f in zip(test_insts, bor_ks, bor_found)]
    seed_bor.append((np.mean(bor_ks), np.std(bor_ks),
                     np.mean(bor_found)*100, np.mean(bor_bits)))

    f1_ks = [rollout(inst, f1_model) for inst in test_insts]
    f1_found = [inst["gold_rank"] <= k for inst, k in zip(test_insts, f1_ks)]
    f1_bits = [max(0, -math.log2(max(k/inst["N"], 1e-12))) if f else 0.0
               for inst, k, f in zip(test_insts, f1_ks, f1_found)]
    seed_f1.append((np.mean(f1_ks), np.std(f1_ks),
                    np.mean(f1_found)*100, np.mean(f1_bits)))

    bucket_data = {}
    for bucket in ["1_easy", "2_medium", "3_hard", "4_vhard"]:
        sub_idx = [i for i, inst in enumerate(test_insts) if inst["bucket"] == bucket]
        if not sub_idx: continue
        bks = [bor_ks[i] for i in sub_idx]
        bfnd = [bor_found[i] for i in sub_idx]
        bucket_data[bucket] = (np.mean(bks), np.mean(bfnd)*100, len(sub_idx))
    seed_bor_buckets.append(bucket_data)

print(f"\n  Done in {time.time()-t0:.0f}s")

# ── Report ──
print("\n" + "=" * 75)
print(f"  BFCL EMBED SCORER: {N_FULL} tools, N={CAND_N}, "
      f"{len(test_insts)} test, {len(SEEDS)} seeds")
print("=" * 75)

print(f"\n{'Method':<15} | {'K':>10} | {'K std':>10} | {'Found%':>12} | {'BoR bits':>12}")
print("-" * 70)

bk = [s[0] for s in seed_bor]; bks = [s[1] for s in seed_bor]
bf = [s[2] for s in seed_bor]; bb = [s[3] for s in seed_bor]
print(f"{'BoR DQN':<15} | {np.mean(bk):>5.1f}±{np.std(bk):.1f} | "
      f"{np.mean(bks):>5.2f}±{np.std(bks):.2f} | "
      f"{np.mean(bf):>5.1f}±{np.std(bf):.1f} | "
      f"{np.mean(bb):>5.2f}±{np.std(bb):.2f}")

fk = [s[0] for s in seed_f1]; fks = [s[1] for s in seed_f1]
ff = [s[2] for s in seed_f1]; fb = [s[3] for s in seed_f1]
print(f"{'F1 DQN':<15} | {np.mean(fk):>5.1f}±{np.std(fk):.1f} | "
      f"{np.mean(fks):>5.2f}±{np.std(fks):.2f} | "
      f"{np.mean(ff):>5.1f}±{np.std(ff):.1f} | "
      f"{np.mean(fb):>5.2f}±{np.std(fb):.2f}")

for fk_val, (k, k_std, found, bor) in sorted(fk_results.items()):
    print(f"{'FK=' + str(fk_val):<15} | {k:>5.1f}     | {k_std:>5.2f}     | "
          f"{found:>5.1f}      | {bor:>5.2f}")

print("-" * 70)

print(f"\n  BoR per bucket (mean±std across {len(SEEDS)} seeds):")
all_buckets = sorted(set(b for bd in seed_bor_buckets for b in bd.keys()))
for bucket in all_buckets:
    ks = [bd[bucket][0] for bd in seed_bor_buckets if bucket in bd]
    fnds = [bd[bucket][1] for bd in seed_bor_buckets if bucket in bd]
    n = seed_bor_buckets[0].get(bucket, (0, 0, 0))[2] if seed_bor_buckets else 0
    if ks:
        print(f"    {bucket:>10}: K={np.mean(ks):.1f}±{np.std(ks):.1f}, "
              f"found={np.mean(fnds):.1f}±{np.std(fnds):.1f}%, n={n}")

print("\nValues after ± are std across 3 seeds.")
print("Done.")

[1/3] Loading BFCL...
  370 tools, 400 queries
[2/3] Embedding...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Embed found@1: 73.2%
  N=370, Train: 280, Test: 120
  Buckets: {'1_easy': 93, '2_medium': 24, '3_hard': 3}

  Fixed-K baselines:
    FK= 1: Found= 77.5%  BoR= 6.61
    FK= 3: Found= 95.0%  BoR= 6.60
    FK= 5: Found= 97.5%  BoR= 6.05
    FK=10: Found= 99.2%  BoR= 5.17
    FK=20: Found=100.0%  BoR= 4.21
    FK=50: Found=100.0%  BoR= 2.89

[3/3] Training BoR + F1 (3 seeds)...

  Seed 42:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  Seed 123:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  Seed 456:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  Done in 184s

  BFCL EMBED SCORER: 370 tools, N=370, 120 test, 3 seeds

Method          |          K |      K st

In [4]:
# ══════════════════════════════════════════════════════════════════════
# ToolBench Main-Table Numbers with 3-Seed Std
# ──────────────────────────────────────────────────────────────────────
# Downloads from Google Drive. Builds tool registry FROM queries
# (matching the HuggingFace mirror's filtering). Self-contained.
# Runtime: ~5 min download + ~20 min training.
# ══════════════════════════════════════════════════════════════════════

SEEDS = [42, 123, 456]
CAND_N = 50
MAX_QUERIES = 2000
TRAIN_EPS = 8000

import sys, subprocess, importlib.util, os, glob
def ensure(name, pip=None):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip or name])
ensure("numpy"); ensure("torch")
ensure("sentence_transformers", "sentence-transformers")
ensure("sklearn", "scikit-learn")
ensure("gdown")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "sympy>=1.13.1"])

import json, math, random, time
from collections import deque, defaultdict
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
import gdown

device = "cuda" if torch.cuda.is_available() else "cpu"

# ══════════════════════════════════════════════════════════════════════
# PART 1: Download
# ══════════════════════════════════════════════════════════════════════
print("[1/5] Downloading ToolBench...", flush=True)

DATA_DIR = "toolbench_data"
if not os.path.exists(DATA_DIR):
    print("  Downloading from Google Drive...", flush=True)
    gdown.download_folder(
        "https://drive.google.com/drive/folders/1TysbSWYpP8EioFu9xPJtpbJZMLLmwAmL",
        quiet=False, output="toolbench_gdrive"
    )
    # Find and unzip data.zip
    for f in ["toolbench_gdrive/data.zip", "data.zip"]:
        if os.path.exists(f):
            subprocess.check_call(["unzip", "-q", "-o", f, "-d", DATA_DIR])
            print(f"  Unzipped {f}")
            break
    assert os.path.exists(DATA_DIR), "Download failed. See manual instructions in earlier cell."

data_root = DATA_DIR
for candidate in [os.path.join(DATA_DIR, "data"), DATA_DIR]:
    if os.path.exists(os.path.join(candidate, "toolenv")):
        data_root = candidate
        break
print(f"  Data root: {data_root}")

# ══════════════════════════════════════════════════════════════════════
# PART 2: Parse - build registry FROM queries (not from toolenv scan)
# ══════════════════════════════════════════════════════════════════════
print("[2/5] Parsing ToolBench...", flush=True)

# First, load ALL instruction entries to find which tools are in the benchmark
all_entries = []
for subdir in ["instruction", "test_instruction"]:
    idir = os.path.join(data_root, subdir)
    if not os.path.exists(idir): continue
    for qf in sorted(glob.glob(os.path.join(idir, "G1_*.json"))):
        try:
            with open(qf) as f:
                data = json.load(f)
            if isinstance(data, dict):
                for qid, entry in data.items():
                    if isinstance(entry, dict):
                        entry["_source"] = os.path.basename(qf)
                        all_entries.append(entry)
            elif isinstance(data, list):
                for entry in data:
                    if isinstance(entry, dict):
                        entry["_source"] = os.path.basename(qf)
                        all_entries.append(entry)
            print(f"  {subdir}/{os.path.basename(qf)}: +{len(data) if isinstance(data, (dict, list)) else 0}")
        except Exception as e:
            print(f"  FAILED {qf}: {e}")

print(f"  Total entries: {len(all_entries)}")

# Extract tool names from api_list in each entry (this is how the HF mirror filtered)
tool_names_in_queries = set()
for entry in all_entries:
    api_list = entry.get("api_list", [])
    if isinstance(api_list, str):
        try: api_list = json.loads(api_list)
        except: continue
    if not isinstance(api_list, list): continue
    for api in api_list:
        if isinstance(api, dict):
            tn = (api.get("tool_name", "") or "").strip()
            if tn: tool_names_in_queries.add(tn)

print(f"  Tools referenced in queries: {len(tool_names_in_queries)}")

# Now build descriptions for ONLY those tools, using toolenv JSONs
toolenv_dir = os.path.join(data_root, "toolenv", "tools")
tool_registry = defaultdict(list)

for cat_dir in glob.glob(os.path.join(toolenv_dir, "*")):
    if not os.path.isdir(cat_dir): continue
    cat_name = os.path.basename(cat_dir)
    for tool_json in glob.glob(os.path.join(cat_dir, "*.json")):
        tool_name = os.path.splitext(os.path.basename(tool_json))[0]
        if tool_name not in tool_names_in_queries:
            continue  # Skip tools not in benchmark queries
        try:
            with open(tool_json) as f:
                td = json.load(f)
            tool_desc = td.get("tool_description", "")
            api_list = td.get("api_list", [])
            for api in api_list:
                api_name = api.get("name", "")
                api_desc = api.get("description", "")
                parts = [p for p in [tool_name, api_name, api_desc, cat_name] if p]
                tool_registry[tool_name].append(" ".join(parts))
            if not tool_registry[tool_name] and tool_desc:
                tool_registry[tool_name].append(f"{tool_name} {tool_desc} {cat_name}")
        except: pass

# For tools in queries but missing from toolenv, use api_list descriptions directly
for entry in all_entries:
    api_list = entry.get("api_list", [])
    if isinstance(api_list, str):
        try: api_list = json.loads(api_list)
        except: continue
    if not isinstance(api_list, list): continue
    for api in api_list:
        if isinstance(api, dict):
            tn = (api.get("tool_name", "") or "").strip()
            if tn and tn not in tool_registry:
                api_name = api.get("api_name", "")
                api_desc = api.get("api_description", "")
                cat = api.get("category_name", "")
                parts = [p for p in [tn, api_name, api_desc, cat] if p]
                if parts:
                    tool_registry[tn].append(" ".join(parts))

tool_descs_map = {}
for tn, texts in tool_registry.items():
    tool_descs_map[tn] = " | ".join(list(dict.fromkeys(texts))[:10])
tool_names_list = sorted(tool_descs_map.keys())
tool_descs_list = [tool_descs_map[n] for n in tool_names_list]
tool_to_idx = {n: i for i, n in enumerate(tool_names_list)}
N_FULL = len(tool_names_list)
print(f"  Filtered tool registry: {N_FULL} tools")

# Build single-tool query list
query_list = []
for entry in all_entries:
    q = str(entry.get("query", "")).strip()
    if not q or len(q) < 10: continue
    api_list = entry.get("api_list", [])
    if isinstance(api_list, str):
        try: api_list = json.loads(api_list)
        except: continue
    if not isinstance(api_list, list): continue
    gt_tools = set()
    for api in api_list:
        if isinstance(api, dict):
            tn = (api.get("tool_name", "") or "").strip()
            if tn in tool_to_idx: gt_tools.add(tn)
    if len(gt_tools) == 1:
        query_list.append({"query": q[:500], "gt_tool": list(gt_tools)[0]})

# Deduplicate
seen = set()
deduped = []
for q in query_list:
    key = q["query"][:100]
    if key not in seen:
        seen.add(key)
        deduped.append(q)
query_list = deduped

random.seed(SEEDS[0])
if len(query_list) > MAX_QUERIES:
    random.shuffle(query_list)
    query_list = query_list[:MAX_QUERIES]

print(f"  Single-tool queries: {len(query_list)}")
assert len(query_list) >= 50, f"Only {len(query_list)} queries. Parsing may need adjustment."

# ══════════════════════════════════════════════════════════════════════
# PART 3: Embed + candidate sets
# ══════════════════════════════════════════════════════════════════════
print("[3/5] Embedding...", flush=True)
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
tool_embs = encoder.encode(tool_descs_list, batch_size=64,
                           show_progress_bar=True, normalize_embeddings=True)
q_embs = encoder.encode([q["query"] for q in query_list], batch_size=64,
                        show_progress_bar=True, normalize_embeddings=True)
score_matrix = q_embs @ tool_embs.T

def difficulty_bucket(rank):
    if rank == 1: return "1_easy"
    if 2 <= rank <= 5: return "2_medium"
    if 6 <= rank <= 20: return "3_hard"
    return "4_vhard"

base_instances = []
for i, q in enumerate(query_list):
    gold_idx = tool_to_idx[q["gt_tool"]]
    ranked_global = np.argsort(-score_matrix[i])
    gold_rank_full = int(np.where(ranked_global == gold_idx)[0][0] + 1)
    base_instances.append({"query": q["query"], "gold_idx": gold_idx,
        "scores_all": score_matrix[i], "ranked_global": ranked_global,
        "gold_rank_full": gold_rank_full, "gt_tool": q["gt_tool"]})

indices = np.arange(len(base_instances))
train_idx, test_idx = train_test_split(indices, test_size=0.30, random_state=SEEDS[0])

CAND_N = min(CAND_N, N_FULL)

def make_cand(base, N):
    gold_idx = base["gold_idx"]
    hard = [j for j in base["ranked_global"] if j != gold_idx][:N-1]
    cand = [gold_idx] + list(hard)
    scores = np.array([base["scores_all"][j] for j in cand], dtype=np.float32)
    order = np.argsort(-scores)
    ranked_ids = [cand[j] for j in order]
    ranked_sc = scores[order]
    gold_rank = int(ranked_ids.index(gold_idx) + 1)
    return {"gold_rank": gold_rank, "ranked_tool_ids": ranked_ids,
            "scores": ranked_sc, "N": N, "query": base["query"],
            "bucket": difficulty_bucket(gold_rank), "gt_tool": base["gt_tool"]}

instances = [make_cand(base_instances[i], CAND_N) for i in range(len(base_instances))]
train_insts = [instances[i] for i in train_idx]
test_insts = [instances[i] for i in test_idx]
print(f"  N={CAND_N}, Train: {len(train_insts)}, Test: {len(test_insts)}")
buckets = defaultdict(int)
for inst in test_insts: buckets[inst["bucket"]] += 1
print(f"  Buckets: {dict(sorted(buckets.items()))}")

# ══════════════════════════════════════════════════════════════════════
# PART 4: Train 3 seeds
# ══════════════════════════════════════════════════════════════════════
print(f"[4/5] Training ({len(SEEDS)} seeds)...", flush=True)

BATCH = 128; REPLAY_SZ = 50000; LR = 1e-3; GAMMA = 0.95; STEP_COST = 0.01
TARGET_EVERY = 500

def bor_reward(success, k, N):
    return -math.log2(max(k / N, 1e-12)) if success else 0.0

def f1_reward(success, k):
    return (2.0 / (k + 1.0)) if success else 0.0

def state_vec(inst, k):
    s = inst["scores"]; N = inst["N"]; idx = min(k-1, N-1)
    cur = float(s[idx])
    nxt = float(s[idx+1]) if idx+1 < N else float(s[idx])
    first = float(s[0]); mean = float(s.mean()); std = float(s.std() + 1e-6)
    gap = cur - nxt if idx+1 < N else 0.0
    return np.array([k/N, math.log2(k+1)/math.log2(N+1), cur, nxt, gap,
                     (cur-mean)/std, cur/(abs(first)+1e-6)], dtype=np.float32)

class QNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7,64), nn.ReLU(),
            nn.Linear(64,64), nn.ReLU(),
            nn.Linear(64,2))
    def forward(self, x): return self.net(x)

def train_dqn(train_insts, reward_name, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    rfn = bor_reward if reward_name == "bor" else f1_reward
    net = QNet().to(device); tgt = QNet().to(device)
    tgt.load_state_dict(net.state_dict())
    opt = optim.Adam(net.parameters(), lr=LR)
    replay = deque(maxlen=REPLAY_SZ)
    eps_s, eps_e = 1.0, 0.05; step = 0
    for ep in range(1, TRAIN_EPS+1):
        inst = random.choice(train_insts); k = 1; done = False
        eps = eps_e + (eps_s - eps_e) * max(0, 1 - ep/TRAIN_EPS)
        while not done:
            sv = state_vec(inst, k)
            if random.random() < eps: a = random.randint(0,1)
            else:
                with torch.no_grad():
                    a = int(net(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
            if k >= inst["N"]: a = 0
            if a == 0:
                success = inst["gold_rank"] <= k
                r = rfn(success, k, inst["N"]) if reward_name == "bor" else rfn(success, k)
                replay.append((sv, a, float(r), None, 1.0)); done = True
            else:
                k2 = k + 1
                if k2 >= inst["N"]:
                    success = inst["gold_rank"] <= inst["N"]
                    r = rfn(success, inst["N"], inst["N"]) if reward_name == "bor" else rfn(success, inst["N"])
                    replay.append((sv, a, float(r), None, 1.0)); done = True
                else:
                    sv2 = state_vec(inst, k2)
                    replay.append((sv, a, -STEP_COST, sv2, 0.0)); k = k2
            step += 1
            if len(replay) >= BATCH:
                batch = random.sample(replay, BATCH)
                states = torch.tensor(np.stack([b[0] for b in batch]),
                                      dtype=torch.float32, device=device)
                actions = torch.tensor([b[1] for b in batch],
                                       dtype=torch.long, device=device).unsqueeze(1)
                rewards = torch.tensor([b[2] for b in batch],
                                       dtype=torch.float32, device=device)
                dones = torch.tensor([b[4] for b in batch],
                                     dtype=torch.float32, device=device)
                nf = torch.tensor([b[3] is not None for b in batch],
                                  dtype=torch.bool, device=device)
                nq = torch.zeros(BATCH, dtype=torch.float32, device=device)
                if nf.any():
                    ns = torch.tensor(
                        np.stack([b[3] for b in batch if b[3] is not None]),
                        dtype=torch.float32, device=device)
                    with torch.no_grad(): nq[nf] = tgt(ns).max(1).values
                qv = net(states).gather(1, actions).squeeze(1)
                tv = rewards + (1-dones) * GAMMA * nq
                loss = nn.SmoothL1Loss()(qv, tv)
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(net.parameters(), 5.0); opt.step()
            if step % TARGET_EVERY == 0:
                tgt.load_state_dict(net.state_dict())
        if ep % 2000 == 0:
            print(f"      {reward_name} {ep}/{TRAIN_EPS}", flush=True)
    return net

def rollout(inst, model):
    k = 1
    while True:
        sv = state_vec(inst, k)
        with torch.no_grad():
            a = int(model(torch.tensor(sv, device=device).unsqueeze(0)).argmax(1).item())
        if a == 0 or k >= inst["N"]: break
        k += 1
    return k

# Fixed-K baselines
print("\n  Fixed-K baselines:")
fk_results = {}
for fk_val in [1, 3, 5, 10, 20, 50]:
    if fk_val > CAND_N: break
    found = np.mean([inst["gold_rank"] <= fk_val for inst in test_insts]) * 100
    bor_vals = [max(0, -math.log2(max(fk_val/inst["N"], 1e-12)))
                if inst["gold_rank"] <= fk_val else 0.0 for inst in test_insts]
    bor = np.mean(bor_vals)
    fk_results[fk_val] = (fk_val, 0, found, bor)
    print(f"    FK={fk_val:>2}: Found={found:>5.1f}%  BoR={bor:>5.2f}")

# 3-seed training
t0 = time.time()
seed_bor = []; seed_f1 = []; seed_bor_buckets = []

for seed in SEEDS:
    print(f"\n  Seed {seed}:", flush=True)
    bor_model = train_dqn(train_insts, "bor", seed)
    f1_model = train_dqn(train_insts, "f1", seed)

    bor_ks = [rollout(inst, bor_model) for inst in test_insts]
    bor_found = [inst["gold_rank"] <= k for inst, k in zip(test_insts, bor_ks)]
    bor_bits = [max(0, -math.log2(max(k/inst["N"], 1e-12))) if f else 0.0
                for inst, k, f in zip(test_insts, bor_ks, bor_found)]
    seed_bor.append((np.mean(bor_ks), np.std(bor_ks),
                     np.mean(bor_found)*100, np.mean(bor_bits)))

    f1_ks = [rollout(inst, f1_model) for inst in test_insts]
    f1_found = [inst["gold_rank"] <= k for inst, k in zip(test_insts, f1_ks)]
    f1_bits = [max(0, -math.log2(max(k/inst["N"], 1e-12))) if f else 0.0
               for inst, k, f in zip(test_insts, f1_ks, f1_found)]
    seed_f1.append((np.mean(f1_ks), np.std(f1_ks),
                    np.mean(f1_found)*100, np.mean(f1_bits)))

    bucket_data = {}
    for bucket in ["1_easy", "2_medium", "3_hard", "4_vhard"]:
        sub_idx = [i for i, inst in enumerate(test_insts) if inst["bucket"] == bucket]
        if not sub_idx: continue
        bks = [bor_ks[i] for i in sub_idx]
        bfnd = [bor_found[i] for i in sub_idx]
        bucket_data[bucket] = (np.mean(bks), np.mean(bfnd)*100, len(sub_idx))
    seed_bor_buckets.append(bucket_data)

print(f"\n  All training done in {time.time()-t0:.0f}s")

# ══════════════════════════════════════════════════════════════════════
# PART 5: Report
# ══════════════════════════════════════════════════════════════════════
print("\n" + "=" * 75)
print(f"  TOOLBENCH: {N_FULL} tools, N={CAND_N}, "
      f"{len(test_insts)} test, {len(SEEDS)} seeds")
print("=" * 75)

print(f"\n{'Method':<15} | {'K':>10} | {'K std':>10} | {'Found%':>12} | {'BoR bits':>12}")
print("-" * 70)

bk = [s[0] for s in seed_bor]; bks = [s[1] for s in seed_bor]
bf = [s[2] for s in seed_bor]; bb = [s[3] for s in seed_bor]
print(f"{'BoR DQN':<15} | {np.mean(bk):>5.1f}±{np.std(bk):.1f} | "
      f"{np.mean(bks):>5.2f}±{np.std(bks):.2f} | "
      f"{np.mean(bf):>5.1f}±{np.std(bf):.1f} | "
      f"{np.mean(bb):>5.2f}±{np.std(bb):.2f}")

fk_m = [s[0] for s in seed_f1]; fk_s = [s[1] for s in seed_f1]
ff = [s[2] for s in seed_f1]; fb = [s[3] for s in seed_f1]
print(f"{'F1 DQN':<15} | {np.mean(fk_m):>5.1f}±{np.std(fk_m):.1f} | "
      f"{np.mean(fk_s):>5.2f}±{np.std(fk_s):.2f} | "
      f"{np.mean(ff):>5.1f}±{np.std(ff):.1f} | "
      f"{np.mean(fb):>5.2f}±{np.std(fb):.2f}")

for fk_val, (k, k_std, found, bor) in sorted(fk_results.items()):
    print(f"{'FK=' + str(fk_val):<15} | {k:>5.1f}     | {k_std:>5.2f}     | "
          f"{found:>5.1f}      | {bor:>5.2f}")

print("-" * 70)

print(f"\n  BoR per bucket (mean±std across {len(SEEDS)} seeds):")
all_buckets = sorted(set(b for bd in seed_bor_buckets for b in bd.keys()))
for bucket in all_buckets:
    ks = [bd[bucket][0] for bd in seed_bor_buckets if bucket in bd]
    fnds = [bd[bucket][1] for bd in seed_bor_buckets if bucket in bd]
    n = seed_bor_buckets[0].get(bucket, (0, 0, 0))[2]
    if ks:
        print(f"    {bucket:>10}: K={np.mean(ks):.1f}±{np.std(ks):.1f}, "
              f"found={np.mean(fnds):.1f}±{np.std(fnds):.1f}%, n={n}")

print("\nValues after ± are std across 3 seeds.")

with open("toolbench_3seed_results.json", "w") as f:
    json.dump({"seed_bor": seed_bor, "seed_f1": seed_f1,
               "fk_results": {str(k): v for k, v in fk_results.items()},
               "seed_bor_buckets": seed_bor_buckets,
               "N_FULL": N_FULL, "n_queries": len(query_list)},
              f, indent=2, default=str)
print("Saved to toolbench_3seed_results.json")
print("Done.")

[1/5] Downloading ToolBench...


Retrieving folder contents


Processing file 1ceLQ9S1IkFTiWeJ3G1FArsD4zY6WYiLa data.zip
Processing file 1rl4b09G7LncVGw1HPCIuLjjMFM4-gKJV reproduction_data.zip


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1ceLQ9S1IkFTiWeJ3G1FArsD4zY6WYiLa
From (redirected): https://drive.google.com/uc?id=1ceLQ9S1IkFTiWeJ3G1FArsD4zY6WYiLa&confirm=t&uuid=f7d478e3-ddef-43fe-b155-dbbadfdf7e33
To: /content/toolbench_gdrive/data.zip
100%|██████████| 1.76G/1.76G [00:32<00:00, 54.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1rl4b09G7LncVGw1HPCIuLjjMFM4-gKJV
From (redirected): https://drive.google.com/uc?id=1rl4b09G7LncVGw1HPCIuLjjMFM4-gKJV&confirm=t&uuid=78c5224a-7f66-4697-8092-16095008fd8a
To: /content/toolbench_gdrive/reproduction_data.zip
100%|██████████| 110M/110M [00:01<00:00, 59.9MB/s]
Download completed


  Unzipped toolbench_gdrive/data.zip
  Data root: toolbench_data/data
[2/5] Parsing ToolBench...
  instruction/G1_query.json: +88995
  test_instruction/G1_category.json: +200
  test_instruction/G1_instruction.json: +200
  test_instruction/G1_tool.json: +200
  Total entries: 89595
  Tools referenced in queries: 3251
  Filtered tool registry: 3251 tools
  Single-tool queries: 2000
[3/5] Embedding...


...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

  N=50, Train: 1400, Test: 600
  Buckets: {'1_easy': 272, '2_medium': 116, '3_hard': 76, '4_vhard': 136}
[4/5] Training (3 seeds)...

  Fixed-K baselines:
    FK= 1: Found= 45.3%  BoR= 2.56
    FK= 3: Found= 58.5%  BoR= 2.37
    FK= 5: Found= 64.7%  BoR= 2.15
    FK=10: Found= 72.3%  BoR= 1.68
    FK=20: Found= 77.3%  BoR= 1.02
    FK=50: Found=100.0%  BoR= 0.00

  Seed 42:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  Seed 123:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  Seed 456:
      bor 2000/8000
      bor 4000/8000
      bor 6000/8000
      bor 8000/8000
      f1 2000/8000
      f1 4000/8000
      f1 6000/8000
      f1 8000/8000

  All training done in 391s

  TOOLBENCH: 3251 tools, N=50, 600 test, 3 seeds

Method          |          K |      K std |      